# NOTEBOOK FEATURE ENGINEERING

In [6]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import chi2_contingency
from itertools import combinations
from scipy.stats import f_oneway

import os
import re

from IPython.display import display, Markdown

import missingno as msno
import sys

from rapidfuzz import process, fuzz

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from itertools import product

import warnings
warnings.filterwarnings('ignore')

In [7]:
df = pd.read_csv(r"C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_eda.csv")

In [8]:
df.columns

Index(['sex', 'race', 'age', 'age_cat', 'decile_score', 'v_decile_score',
       'is_recid', 'is_violent_recid', 'score_text', 'screening_date',
       'two_year_recid', 'priors_count', 'c_charge_degree', 'start', 'end',
       'event', 'c_jail_in', 'c_jail_out', 'juv_fel_count', 'juv_misd_count',
       'juv_other_count', 'person_id', 'agency_text', 'maritalstatus',
       'language', 'rawscore', 'days_in_jail', 'juv_priors_count'],
      dtype='object')

In [9]:
lista_variables_modelo = [
    'person_id',
    'decile_score',
    'rawscore',
    'v_decile_score',
    'is_recid',
    'is_violent_recid',
    'two_year_recid',
    'sex',
    'race',
    'age',
    'days_in_jail',
    'priors_count',
    'juv_priors_count',
    'c_charge_degree',
    'maritalstatus',
    'juv_fel_count',
    'juv_misd_count',
    'juv_other_count'
]

In [10]:
df = df[lista_variables_modelo]

In [11]:
df.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age',
       'days_in_jail', 'priors_count', 'juv_priors_count', 'c_charge_degree',
       'maritalstatus', 'juv_fel_count', 'juv_misd_count', 'juv_other_count'],
      dtype='object')

In [12]:
df['race'] = df['race'].replace({'asian': 'other', 'native american': 'other'})

In [13]:
df.race.value_counts()

race
african-american    2944
caucasian           2031
hispanic             512
other                363
Name: count, dtype: int64

In [14]:
df["two_year_recid"].sum()

2181

In [15]:
df.shape

(5850, 18)

In [16]:
df['sex'] = df['sex'].replace({'male': 0, 'female': 1})

In [17]:
df.sex.value_counts()

sex
0    4719
1    1131
Name: count, dtype: int64

In [18]:
df['c_charge_degree'] = df['c_charge_degree'].replace({'felony': 0, 'misdemeanor': 1})

In [19]:
df.c_charge_degree.value_counts()

c_charge_degree
0    3731
1    2119
Name: count, dtype: int64

In [20]:
df['maritalstatus'] = df['maritalstatus'].replace({'married': 'significant other', 'divorced': 'separated', 'widowed': 'other', 'unknown': 'other'})

In [21]:
df.maritalstatus.value_counts()

maritalstatus
single               4531
significant other     891
separated             376
other                  52
Name: count, dtype: int64

In [22]:
def one_hot_encoding(df, column, drop_val):
    encoder = OneHotEncoder(
    drop=[drop_val], 
    sparse_output=False
    )

    encoded = encoder.fit_transform(df[[column]])

    df_encoded = pd.DataFrame(
    encoded,
    columns = encoder.get_feature_names_out([column])
    )

    return df_encoded

In [23]:
df_model = pd.concat([df, one_hot_encoding(df, 'maritalstatus', 'single')], axis = 1)

In [24]:
#df_no_caucasian = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'caucasian')], axis = 1)

In [25]:
#df_model = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'african-american')], axis = 1)

In [26]:
df_model['priors_freq']=(df_model['priors_count']/(df_model['age']-18))

In [27]:
df_model.priors_freq.max()

4.5

In [28]:
df_model['priors_freq_2']=((df_model['priors_count']+(df_model['priors_count'].mean()))/((df_model['age']-18)+(df_model['age']-18).mean()))

In [29]:
df_model.priors_count.mean()

3.235213675213675

In [30]:
(df_model['age']-18).mean()

17.12205128205128

In [31]:
tasa_media = df_model.priors_count.sum()/(df_model['age']-18).sum()

In [32]:
tasa_media

0.1889501218002476

In [33]:
beta = 5

In [34]:
alfa = tasa_media * beta

In [35]:
df_model['priors_freq_3']=((df_model['priors_count']+ alfa)/((df_model['age']-18)+ beta))

In [36]:
display(df_model.priors_freq_2.max())
display(df_model.priors_freq_3.max())

1.1574593950816026

1.99605361492866

In [37]:
df_model.head()

,person_id,decile_score,rawscore,v_decile_score,is_recid,is_violent_recid,two_year_recid,sex,race,age,...,maritalstatus,juv_fel_count,juv_misd_count,juv_other_count,maritalstatus_other,maritalstatus_separated,maritalstatus_significant other,priors_freq,priors_freq_2,priors_freq_3
0,62384.0,2,-3.03,1,1,0,1,0,hispanic,96,...,other,0,0,0,1.0,0.0,0.0,0.025641,0.055037,0.035479
1,50959.0,1,-4.50,1,0,0,0,0,hispanic,83,...,significant other,0,0,0,0.0,0.0,1.0,0.000000,0.039395,0.013496
2,53038.0,1,-4.63,1,0,0,0,0,caucasian,80,...,significant other,0,0,0,0.0,0.0,1.0,0.000000,0.040889,0.014101
3,56006.0,1,-4.05,1,0,0,0,0,caucasian,79,...,separated,0,0,0,0.0,1.0,0.0,0.016393,0.054213,0.029466
4,57794.0,1,-3.23,1,0,0,0,0,african-american,77,...,significant other,0,0,0,0.0,0.0,1.0,0.000000,0.042500,0.014762


In [38]:
df_model.to_csv(r'C:\Users\JAIME\Documents\Github\TFM-Sesgos-en-el-sistema-judicial-de-EEUU-\00_Data\00_Processed\df_modelo.csv', index=False)